# Neural Models — Learning, Depth, Activations, and Output Layers

**Lab Exercise.** A tiny XOR problem makes the *representation* and *learning mechanism* of a
neural network transparent enough to inspect.

Covered here, in order:

* **Task 1** — understand the problem before coding (linear separability).
* **Task 2** — design the agent (a 2–2–1 network).
* **Task 3** — "LLM"-generated minimal PyTorch implementation (inspected before running).
* **Task 4** — execute, test, and diagnose: basic learning, backprop check, **symmetry**
  experiment, and an **activation** experiment (sigmoid/tanh/ReLU).
* **Task 5** — extend to a three-class softmax problem; explain the $p-y$ logit gradient.
* **Reflection questions.**

> Use the LLM as a collaborator for *engineering*, not a substitute for the *science* that
> specifies what behaviour the model should exhibit and what evidence supports it.


## Motivating scenario — redundant safety sensors (XOR)

Two binary sensors $x_1,x_2$. Raise a *disagreement* warning exactly when one is active and the
other is not — the XOR function:

| $x_1$ | $x_2$ | $y$ |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |


## Task 1 — Understand the problem before coding

* **Input space** $\mathcal{X}=\{0,1\}^2$; **output space** $\mathcal{Y}=\{0,1\}$.
* **Labelled examples:** $(0,0)\!\to\!0,\ (0,1)\!\to\!1,\ (1,0)\!\to\!1,\ (1,1)\!\to\!0$.

Points in the plane (● = class 1 "warning", ○ = class 0):

```
 x2
  1 |  ●(0,1)        ○(1,1)
    |
  0 |  ○(0,0)        ●(1,0)
    +------------------------ x1
         0              1
```

* **Why one straight line cannot separate the classes:** the two class-1 points $(0,1),(1,0)$ lie
  on one diagonal and the two class-0 points $(0,0),(1,1)$ on the other. Any single straight line
  puts at least one point on the wrong side — XOR is **not linearly separable**.
* **Prediction for a single affine map + sigmoid:** it can only carve the plane with one linear
  boundary, so it **cannot** fit XOR. We expect it to stall near 50% accuracy with loss around
  $\ln 2\approx0.693$, predicting roughly 0.5 everywhere.

> **Think About It:** XOR lets us test a claim about *representation* with only four points — a
> model can have many parameters yet still have the *wrong kind* of representation (purely linear).


In [1]:
import numpy as np
import torch
import torch.nn as nn

# The four XOR examples, used throughout.
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y = torch.tensor([[0.], [1.], [1.], [0.]])
print("X =\n", X)
print("y =", Y.view(-1).tolist())

# Confirm the prediction: a single affine layer + sigmoid cannot separate XOR.
torch.manual_seed(0)
linear = nn.Linear(2, 1)
opt = torch.optim.Adam(linear.parameters(), lr=0.1)
lossf = nn.BCEWithLogitsLoss()
for _ in range(2000):
    opt.zero_grad(); L = lossf(linear(X), Y); L.backward(); opt.step()
probs = torch.sigmoid(linear(X)).view(-1)
print(f"\nLinear-only model: final loss = {L.item():.4f} (~0.69 = ln 2 means no better than chance)")
print("probabilities:", [round(p, 3) for p in probs.tolist()], "-> cannot separate XOR")


X =
 tensor([[0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]])
y = [0.0, 1.0, 1.0, 0.0]



Linear-only model: final loss = 0.6931 (~0.69 = ln 2 means no better than chance)
probabilities: [0.5, 0.5, 0.5, 0.5] -> cannot separate XOR


## Task 2 — Design the intelligent agent

Baseline: **2 inputs → 2 hidden units → 1 output**, with

* a **nonlinear hidden activation** (sigmoid/tanh, later ReLU);
* a **sigmoid output** (one yes/no answer) — in PyTorch we keep raw **logits** and use
  `BCEWithLogitsLoss` (numerically stabler than `Sigmoid` + `BCELoss`);
* **binary cross-entropy** loss; **gradient-based** optimisation.

**1. Why is the hidden nonlinearity scientifically necessary?** A stack of affine layers with no
nonlinearity collapses to a single affine map ($W_2(W_1x+b_1)+b_2 = W'x+b'$), which can only draw
one linear boundary. The nonlinear hidden layer lets the network bend the input space so the two
XOR classes become separable — depth *without* nonlinearity adds nothing.

**2. Why sigmoid + BCE for the output?** The target is a single probability in $[0,1]$. Sigmoid maps
a logit to that range, and binary cross-entropy is its matching loss: together they give a convex
output-layer objective whose logit gradient is the clean $\hat{p}-y$, so learning signals are
well-scaled.

**3. Evidence that counts as successful learning (validation criteria):**
(i) final loss near 0; (ii) all **four** predictions correct after thresholding at 0.5;
(iii) non-zero gradients flowing to the first layer during training; (iv) the behaviour is
reproducible across seeds that break initial symmetry.

> **Think About It:** the hidden units are never given target values. If the net learns XOR, it is
> **backpropagation** — distributing the output error to each hidden unit via the chain rule — that
> *determines* what each hidden unit should compute.


## Task 3 — "LLM"-generated first implementation

Below is the concise PyTorch the prompt would request: the four XOR examples, a 2–2–1 net, a
chosen hidden activation, logits + `BCEWithLogitsLoss`, random init, a few thousand CPU steps,
and printing of the final loss, four probabilities, thresholded labels, and one gradient tensor.

**Prompt used (recorded):** *"Generate minimal PyTorch code for a 2–2–1 network on the four XOR
examples. Use a nonlinear hidden activation and logits with BCEWithLogitsLoss. Random init, a few
thousand full-batch CPU steps. After training report the final loss, all four probabilities,
thresholded labels, and one parameter-gradient tensor. Set a random seed for reproducibility."*

**Inspect before running** — where each piece lives:
forward pass = `net(X)`; scalar loss = `lossf(logits, Y)`; reverse-mode AD = `loss.backward()`;
parameter update = `opt.step()`.

**Two corrections made to typical generated code:** (1) use `BCEWithLogitsLoss` on raw logits
rather than `Sigmoid` + `BCELoss` (stability); (2) fix the seed so the symmetry/activation
experiments are reproducible.


In [2]:
def make_net(activation="sigmoid", seed=None):
    """2 -> 2 -> 1 network returning raw logits (no final sigmoid).

    Seeding BEFORE construction makes the *random weight initialisation* reproducible --
    essential, since the whole experiment depends on how the weights are initialised.
    """
    if seed is not None:
        torch.manual_seed(seed)
    act = {"sigmoid": nn.Sigmoid, "tanh": nn.Tanh, "relu": nn.ReLU}[activation]
    return nn.Sequential(
        nn.Linear(2, 2),   # hidden layer: W(1) is this layer's weight
        act(),
        nn.Linear(2, 1),   # output layer: produces a logit
    )

def train(net, steps=4000, lr=0.1, record_early_grad=False):
    """Full-batch training with Adam + BCEWithLogitsLoss. Returns a results dict.

    (Adam has no stochasticity here, so reproducibility comes from the init seed in make_net.)
    """
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    lossf = nn.BCEWithLogitsLoss()
    initial_loss = lossf(net(X), Y).item()
    early_grad_norm = None
    for step in range(steps):
        opt.zero_grad()
        logits = net(X)
        loss = lossf(logits, Y)
        loss.backward()
        if record_early_grad and step == 1:
            # Euclidean norm of the first-layer weight gradient early in training.
            early_grad_norm = net[0].weight.grad.norm().item()
        opt.step()
    with torch.no_grad():
        probs = torch.sigmoid(net(X)).view(-1)
        preds = (probs > 0.5).float()
        correct = int((preds == Y.view(-1)).sum().item())
    return {"net": net, "initial_loss": initial_loss, "final_loss": loss.item(),
            "probs": probs.tolist(), "preds": preds.tolist(),
            "correct": correct, "early_grad_norm": early_grad_norm}


## Task 4 — Execute, test, diagnose

### Part A — basic learning check


In [3]:
res = train(make_net("sigmoid", seed=2))
print(f"Initial loss : {res['initial_loss']:.4f}")
print(f"Final loss   : {res['final_loss']:.6f}")
print("Probabilities:", [round(p, 4) for p in res['probs']])
print("Predictions  :", [int(p) for p in res['preds']], " targets:", [int(v) for v in Y.view(-1)])
print(f"Correct      : {res['correct']}/4",
      "=> XOR learned" if res['correct'] == 4 else "=> not yet learned")


Initial loss : 0.6985
Final loss   : 0.000051
Probabilities: [0.0, 0.9999, 0.9999, 0.0001]
Predictions  : [0, 1, 1, 0]  targets: [0, 1, 1, 0]
Correct      : 4/4 => XOR learned


### Part B — backpropagation check

After one forward pass and `backward()`, `parameter.grad` holds $\partial L/\partial W^{(1)}$ — the
gradient of the scalar loss with respect to each first-layer weight, computed by reverse-mode
automatic differentiation (the chain rule). Because the loss is a **mean** over the four examples,
this gradient is the **average** of the four per-example gradients (the derivative of a sum/mean is
the sum/mean of the derivatives).


In [4]:
net = make_net("tanh", seed=2)
lossf = nn.BCEWithLogitsLoss()
loss = lossf(net(X), Y)
loss.backward()
print("dL/dW(1)  (first-layer weight gradient), shape", tuple(net[0].weight.grad.shape), ":")
print(net[0].weight.grad)
print("\ndL/db(1)  (first-layer bias gradient):")
print(net[0].bias.grad)
print("\nGradient is non-zero -> backprop supplies a learning signal to the hidden layer.")

# Confirm the mean-loss gradient equals the average of per-example gradients.
net2 = make_net("tanh"); net2.load_state_dict(net.state_dict())
per_example = torch.zeros_like(net2[0].weight)
for i in range(4):
    net2.zero_grad()
    nn.BCEWithLogitsLoss()(net2(X[i:i+1]), Y[i:i+1]).backward()
    per_example += net2[0].weight.grad
per_example /= 4
print("\nMax |mean-grad - average-of-per-example-grads| =",
      (net[0].weight.grad - per_example).abs().max().item(), "(~0 confirms the averaging)")


dL/dW(1)  (first-layer weight gradient), shape (2, 2) :
tensor([[-0.0006,  0.0005],
        [ 0.0021,  0.0046]])

dL/db(1)  (first-layer bias gradient):
tensor([-0.0005, -0.0013])

Gradient is non-zero -> backprop supplies a learning signal to the hidden layer.

Max |mean-grad - average-of-per-example-grads| = 0.0 (~0 confirms the averaging)


### Part C — symmetry experiment (identical / zero initialisation)

Set **all weights and biases to zero** and train. Identical hidden units compute the same output
and receive the same gradient, so the two rows of the hidden weight matrix stay identical forever —
the units can never differentiate, and the network cannot represent XOR.


In [5]:
def train_zero_init(steps=2000, lr=0.1):
    net = make_net("tanh")
    with torch.no_grad():
        for p in net.parameters():
            p.zero_()                      # destroy all asymmetry
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    lossf = nn.BCEWithLogitsLoss()
    row_diffs = []
    for step in range(steps):
        opt.zero_grad(); loss = lossf(net(X), Y); loss.backward(); opt.step()
        if step in (0, 1, 5, 50, 500, steps - 1):
            w = net[0].weight.detach()
            row_diffs.append((step, (w[0] - w[1]).abs().max().item()))
    probs = torch.sigmoid(net(X)).view(-1)
    correct = int(((probs > 0.5).float() == Y.view(-1)).sum().item())
    return net, row_diffs, loss.item(), correct

znet, diffs, zloss, zcorrect = train_zero_init()
print("Max |row0 - row1| of hidden weights over training:")
for step, d in diffs:
    print(f"  step {step:>4}: {d:.3e}")
print(f"\nFinal loss {zloss:.4f}, correct {zcorrect}/4.")
print("Rows stay identical (difference stays 0) -> hidden units never differentiate -> XOR not learned.")
print("Lesson: random (symmetry-breaking) initialisation is essential.")


Max |row0 - row1| of hidden weights over training:
  step    0: 0.000e+00
  step    1: 0.000e+00
  step    5: 0.000e+00
  step   50: 0.000e+00
  step  500: 0.000e+00
  step 1999: 0.000e+00

Final loss 0.6931, correct 2/4.
Rows stay identical (difference stays 0) -> hidden units never differentiate -> XOR not learned.
Lesson: random (symmetry-breaking) initialisation is essential.


### Part D — activation experiment (sigmoid / tanh / ReLU)

Train the random-initialised network three times (same seed so only the activation changes) and
record final loss, whether all four examples are classified correctly, and the early first-layer
gradient norm.


In [6]:
rows = []
for act in ["sigmoid", "tanh", "relu"]:
    r = train(make_net(act, seed=2), record_early_grad=True)
    rows.append((act, r["final_loss"], r["correct"], r["early_grad_norm"]))

print(f"{'Hidden act':<10}{'Final loss':>13}{'4/4?':>8}{'Early ||grad W(1)||':>22}")
for act, fl, cr, gn in rows:
    print(f"{act:<10}{fl:>13.6f}{('yes' if cr==4 else 'no'):>8}{gn:>22.4f}")


Hidden act   Final loss    4/4?   Early ||grad W(1)||
sigmoid        0.000051     yes                0.0007
tanh           0.000019     yes                0.0085
relu           0.000008     yes                0.0147


**Interpretation (for *this* experiment — not a universal ranking).** With seed 2 all three
activations break symmetry and reach 4/4. The early gradient norms differ by *mechanism*: a
saturated **sigmoid** has small derivatives (its output is squashed into $(0,1)$ and
$\sigma'=\sigma(1-\sigma)\le0.25$), **tanh** is zero-centred with larger mid-range derivatives, and
an active **ReLU** unit passes derivative 1 (but a unit stuck at $\le 0$ passes derivative 0). So
different activations feed back differently-scaled learning signals even on this tiny task. Note
XOR with only 2 hidden units is initialisation-sensitive — some seeds fail to converge — which is
itself a useful observation about optimisation vs representation.

> **Think About It:** a saturated sigmoid (derivative ≈ 0) and a negative ReLU (derivative = 0)
> both produce a small gradient by *different* mechanisms. You can distinguish them by inspecting
> the **pre-activations**: a saturated sigmoid has large-magnitude pre-activations with outputs
> near 0/1; a dead ReLU has negative pre-activations with outputs exactly 0.


## Task 5 — Extend to three classes (softmax)

Same inputs, a three-class decision ($y=x_1+x_2$):

| Class | Meaning | inputs |
|---|---|---|
| 0 | both inactive | (0,0) |
| 1 | disagree | (0,1), (1,0) |
| 2 | both active | (1,1) |

Replace the single output with **three logits** and train with multiclass cross-entropy
(`nn.CrossEntropyLoss`, which applies log-softmax internally).

**Predictions before accepting the change:**
1. **Final weight matrix shape:** the output `Linear(2, 3)` has weight shape **(3, 2)** (3 classes
   × 2 hidden features), plus a length-3 bias.
2. **Logits per example:** **3**.
3. **Why softmax probabilities sum to one:** softmax normalises by dividing each
   $e^{z_k}$ by $\sum_j e^{z_j}$, so the components are non-negative and sum to exactly 1 — a valid
   categorical distribution.
4. **Why the logit gradient is $p-y$:** for softmax + cross-entropy with one-hot target $y$, the
   loss derivative w.r.t. the logits is $\partial L/\partial z = p - y$ (predicted distribution
   minus the one-hot target) — a clean, well-scaled error signal, which is why this output/loss
   pairing is chosen for classification.


In [7]:
Yc = torch.tensor([0, 1, 1, 2])  # class per input row (long tensor for CrossEntropyLoss)

def make_classifier(activation="tanh"):
    act = {"sigmoid": nn.Sigmoid, "tanh": nn.Tanh, "relu": nn.ReLU}[activation]
    return nn.Sequential(nn.Linear(2, 2), act(), nn.Linear(2, 3))  # 3 logits out

torch.manual_seed(2)
clf = make_classifier("tanh")
opt = torch.optim.Adam(clf.parameters(), lr=0.1)
lossf = nn.CrossEntropyLoss()
for _ in range(4000):
    opt.zero_grad(); loss = lossf(clf(X), Yc); loss.backward(); opt.step()

print("Final output weight shape:", tuple(clf[-1].weight.shape), "(expected (3, 2))")
print(f"Final loss: {loss.item():.6f}\n")
with torch.no_grad():
    probs = torch.softmax(clf(X), dim=1)
for i in range(4):
    row = [round(p, 3) for p in probs[i].tolist()]
    print(f"input {X[i].tolist()} -> class probs {row}  "
          f"pred={int(probs[i].argmax())} target={int(Yc[i])}")
print("\nAll four correct:",
      int((probs.argmax(1) == Yc).sum().item()), "/ 4")


Final output weight shape: (3, 2) (expected (3, 2))
Final loss: 0.000005

input [0.0, 0.0] -> class probs [1.0, 0.0, 0.0]  pred=0 target=0
input [0.0, 1.0] -> class probs [0.0, 1.0, 0.0]  pred=1 target=1
input [1.0, 0.0] -> class probs [0.0, 1.0, 0.0]  pred=1 target=1
input [1.0, 1.0] -> class probs [0.0, 0.0, 1.0]  pred=2 target=2

All four correct: 4 / 4


In [8]:
# Verify softmax sums to 1 and is shift-invariant (numerical-stability diagnostic).
with torch.no_grad():
    logits0 = clf(X[1:2])                      # one example
    p0 = torch.softmax(logits0, dim=1)
    p_shift = torch.softmax(logits0 + 100.0, dim=1)   # add constant 100 to all logits
print("softmax(logits)      :", [round(v, 4) for v in p0.view(-1).tolist()],
      " sum =", round(p0.sum().item(), 6))
print("softmax(logits + 100):", [round(v, 4) for v in p_shift.view(-1).tolist()],
      " sum =", round(p_shift.sum().item(), 6))
print("max |difference| =", (p0 - p_shift).abs().max().item(),
      "-> softmax is shift-invariant.")
print("\nStable implementations subtract max(logits) before exp() so that the largest")
print("exponent is e^0 = 1, preventing overflow, without changing the probabilities.")

# p - y for a trained-model example (analytic form of the softmax-cross-entropy logit gradient).
ex = 0
with torch.no_grad():
    p = torch.softmax(clf(X[ex:ex+1]), dim=1)
onehot = torch.zeros(1, 3); onehot[0, Yc[ex]] = 1.0
print(f"\nLogit error signal p - y for input {X[ex].tolist()} (true class {int(Yc[ex])}):")
print("  p     =", [round(v, 4) for v in p.view(-1).tolist()])
print("  p - y =", [round(v, 4) for v in (p - onehot).view(-1).tolist()],
      "(this is exactly dL/dz)")


softmax(logits)      : [0.0, 1.0, 0.0]  sum = 1.0
softmax(logits + 100): [0.0, 1.0, 0.0]  sum = 1.0
max |difference| = 2.2737367544323206e-12 -> softmax is shift-invariant.

Stable implementations subtract max(logits) before exp() so that the largest
exponent is e^0 = 1, preventing overflow, without changing the probabilities.

Logit error signal p - y for input [0.0, 0.0] (true class 0):
  p     = [1.0, 0.0, 0.0]
  p - y = [-0.0, 0.0, 0.0] (this is exactly dL/dz)


The key fact is that `p - y` *is* the gradient of softmax-cross-entropy with respect to the
logits. The cleaner demonstration below verifies it against autograd on a leaf logit tensor.


In [9]:
# Clean p - y demonstration on a leaf logit tensor.
z = torch.tensor([[2.0, 0.5, -1.0]], requires_grad=True)   # arbitrary logits
target = torch.tensor([2])                                 # true class = 2
nn.CrossEntropyLoss()(z, target).backward()
p = torch.softmax(z.detach(), dim=1)
onehot = torch.tensor([[0., 0., 1.]])
print("softmax p      :", [round(v, 4) for v in p.view(-1).tolist()])
print("analytic p - y :", [round(v, 4) for v in (p - onehot).view(-1).tolist()])
print("autograd dL/dz :", [round(v, 4) for v in z.grad.view(-1).tolist()])
print("=> they match: the softmax-cross-entropy logit gradient is exactly p - y.")


softmax p      : [0.7856, 0.1753, 0.0391]
analytic p - y : [0.7856, 0.1753, -0.9609]
autograd dL/dz : [0.7856, 0.1753, -0.9609]
=> they match: the softmax-cross-entropy logit gradient is exactly p - y.


> **Think About It:** next-token prediction is classification over a huge vocabulary. The
> *mathematics* here — softmax over logits, cross-entropy loss, the $p-y$ gradient — stays exactly
> the same when the number of classes grows to tens of thousands. What changes dramatically is the
> *surrounding architecture* (embeddings, deep context-mixing layers, an output projection of size
> vocabulary) and the engineering needed to make it efficient.


## Suggested Result Table (activation experiment)

Filled from the Part D run above:

| Hidden activation | Final loss | 4/4 correct? | Early $\lVert\nabla_{W^{(1)}}L\rVert_2$ |
|---|---|---|---|
| Sigmoid | see Part D output | yes | see Part D output |
| Tanh | see Part D output | yes | see Part D output |
| ReLU | see Part D output | yes | see Part D output |

(The executed numbers are printed by the Part D cell; they are seed-dependent, so the table points
to the live output rather than hard-coding values.)


## Reflection Questions

1. **Depth vs nonlinearity.** XOR showed that stacking affine layers is not enough — without a
   hidden *nonlinearity* the network stays linear and fails, while the same depth *with* a
   nonlinearity succeeds. Representation power comes from nonlinearity, not depth alone.
2. **Evidence of a useful learning signal.** Not just a non-zero gradient, but the loss falling
   from ~0.69 toward 0, all four predictions flipping to correct, and the behaviour reproducing
   across symmetry-breaking seeds — i.e. the gradient pointed *somewhere useful*, not merely
   somewhere.
3. **Zero/identical init.** Identical hidden units compute identical outputs and receive identical
   gradients, so their weight rows update identically and never differentiate — the network is
   stuck in a symmetric subspace that cannot represent XOR.
4. **Activation vs the observed gradient.** *Scientifically*, each activation's derivative profile
   (sigmoid ≤ 0.25 and saturating; tanh zero-centred; ReLU 1 when active, 0 when dead) changes how
   error propagates. *Engineering observation*: the early gradient norms and convergence behaviour
   differed between activations for this specific experiment — which we report without claiming a
   universal "best".
5. **Output layer + loss chosen together.** The output activation and loss must match the task:
   sigmoid+BCE for one probability, softmax+cross-entropy for one-of-$K$. The pairing is what makes
   the logit gradient clean ($\hat p - y$); mismatched pairs give ill-scaled or incorrect gradients.
6. **LLM help vs human verification.** The LLM quickly produced correct PyTorch boilerplate
   (productivity win). Human verification was essential to (a) switch to `BCEWithLogitsLoss`, and
   (b) confirm the *experiment* was right — e.g. that symmetry actually persists under zero init and
   that all four labels are correct — things syntactically-correct code can silently get wrong.
7. **Which tests survive scaling up?** Cheap, property-based checks scale: loss decreasing,
   held-out accuracy, softmax normalisation, shift-invariance, spot gradient checks. Exhaustive
   finite-difference gradient checks over *all* parameters, and enumerating *all* inputs, become
   infeasible and are replaced by sampled/stochastic checks.

> **Responsible use:** LLMs are great for boilerplate, explaining library calls, and first drafts —
> but they can invent APIs, alter requirements, or hide numerical mistakes. You remain responsible
> for the correctness of the experiment, code, and interpretation.
